# Phase 5 — TS 분할 Drive 전처리 노트북

Drive에 업로드된 TS.zXX 파일을 Colab에서 압축 해제 + 전처리하여
`finetune_dataset/`에 누적 저장합니다.

**사용 순서**
1. 로컬에서 TS.zXX 다운로드 → Drive `aihub_food/` 에 업로드
2. TL.zip + VL.zip 도 같은 폴더에 있는지 확인
3. Step 1 셀 실행 (Drive 마운트 + 경로 설정)
4. Step 2~5 순서대로 실행
5. 다음 분할: `TS_ZIP_NAME` 변경 → Step 2부터 재실행

**결과**: `finetune_dataset/train/images+labels`, `val/images+labels` 에 자동 누적
- 이미 처리한 이미지는 중복 스킵됨
- 모든 분할 완료 후 `phase5_finetune_colab.ipynb` Step 3(data.yaml) → Step 4(학습) 실행

## Step 1 — 환경 설정 및 Drive 마운트

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi

In [ ]:
from google.colab import drive
import os, shutil
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/LGHellovision/Project 02/Object Detection'
DATASET_DIR  = f'{DRIVE_ROOT}/aihub_food'      # TL.zip, VL.zip, TS.zXX 위치
FINETUNE_DIR = f'{DRIVE_ROOT}/finetune_dataset' # 전처리 결과 누적 위치

for d in [
    DATASET_DIR,
    f'{FINETUNE_DIR}/train/images', f'{FINETUNE_DIR}/train/labels',
    f'{FINETUNE_DIR}/val/images',   f'{FINETUNE_DIR}/val/labels',
]:
    os.makedirs(d, exist_ok=True)

print('Drive 마운트 완료')
print(f'DATASET_DIR:  {DATASET_DIR}')
print(f'FINETUNE_DIR: {FINETUNE_DIR}')
!df -h /content | tail -1

## Step 2 — 처리할 TS 분할 지정 + 압축 해제

In [ ]:
# ── 여기만 변경 ───────────────────────────────────────────────────────────────
TS_ZIP_NAME = "TS.z02"   # ← TS.z01 / TS.z02 / ... / TS.zip

# ── 경로 설정 ─────────────────────────────────────────────────────────────────
TS_ZIP_DRIVE    = f"{DATASET_DIR}/{TS_ZIP_NAME}"
TS_EXTRACT_LOCAL = "/content/ts_extract"

# 디스크 / 파일 확인
total, used, free = shutil.disk_usage("/content")
print(f"Colab 디스크 여유: {free/1e9:.1f} GB  (100 GB+ 필요)")

if os.path.exists(TS_ZIP_DRIVE):
    size = os.path.getsize(TS_ZIP_DRIVE)
    print(f"Drive 파일: {TS_ZIP_NAME}  ({size/1e9:.1f} GB)")
    if free < size * 1.2:
        print("WARNING: 여유 공간 부족 — 이전 ts_extract 삭제 후 재시도")
else:
    print(f"ERROR: {TS_ZIP_DRIVE} 없음 — Drive > aihub_food/ 에 업로드 확인")

In [ ]:
# p7zip 설치 + 압축 해제
!apt-get install -q p7zip-full

if os.path.exists(TS_EXTRACT_LOCAL):
    shutil.rmtree(TS_EXTRACT_LOCAL)
os.makedirs(TS_EXTRACT_LOCAL, exist_ok=True)

print(f"압축 해제 중: {TS_ZIP_NAME} -> {TS_EXTRACT_LOCAL}")
ret = os.system(f'7z x "{TS_ZIP_DRIVE}" -o"{TS_EXTRACT_LOCAL}" -y')
if ret != 0:
    raise RuntimeError("압축 해제 실패")

imgs = (list(Path(TS_EXTRACT_LOCAL).rglob("*.jpg")) +
        list(Path(TS_EXTRACT_LOCAL).rglob("*.jpeg")) +
        list(Path(TS_EXTRACT_LOCAL).rglob("*.png")))
print(f"완료 — 이미지 {len(imgs):,}장")
!df -h /content | tail -1

## Step 3 — TL.zip / VL.zip 압축 해제 (최초 1회만)

In [ ]:
import zipfile

for zip_name in ['TL.zip', 'VL.zip']:
    zip_path = Path(DATASET_DIR) / zip_name
    if not zip_path.exists():
        print(f"없음: {zip_name} — Drive aihub_food/ 에 업로드 확인")
        continue
    # 이미 해제됐으면 스킵
    json_check = list(Path(DATASET_DIR).rglob('*.json'))
    if json_check:
        print(f"스킵: {zip_name} (이미 해제됨, JSON {len(json_check):,}개)")
        continue
    print(f"압축 해제: {zip_name}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATASET_DIR)
    print(f"  완료")

jsons = list(Path(DATASET_DIR).rglob('*.json'))
print(f"\n총 JSON: {len(jsons):,}개")

## Step 4 — 640x640 리사이즈 + YOLO 변환 -> Drive finetune_dataset/

In [ ]:
import json as _json, cv2, yaml
from collections import defaultdict

TARGET_SIZE  = 640
JPEG_QUALITY = 85

def _collect_classes(label_dir):
    jsons = list(Path(label_dir).rglob("*.json"))
    print(f"  JSON {len(jsons):,}개 스캔 중...")
    cnt = defaultdict(int)
    for jf in jsons:
        try:
            d = _json.loads(jf.read_text(encoding="utf-8"))
            fc = d.get("data", {}).get("food_type", {}).get("fc", "").strip()
            if fc:
                cnt[fc] += 1
        except Exception:
            pass
    names = [c for c, _ in sorted(cnt.items(), key=lambda x: -x[1])]
    print(f"  클래스 {len(names)}종")
    return names, {c: i for i, c in enumerate(names)}

def _build_index(images_dir):
    idx = {}
    for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"):
        for p in Path(images_dir).rglob(ext):
            idx[p.name.lower()] = str(p)
    print(f"  이미지 {len(idx):,}장 인덱싱")
    return idx

def _convert_one(jf, idx, cls2id, out_img, out_lbl):
    try:
        d = _json.loads(Path(jf).read_text(encoding="utf-8")).get("data", {})
        fc = d.get("food_type", {}).get("fc", "").strip()
        if not fc or fc not in cls2id:
            return False
        ii = d.get("image_info", {})
        iw, ih = float(ii.get("width", 0)), float(ii.get("height", 0))
        fname = ii.get("file_name", "")
        if not fname or iw <= 0 or ih <= 0:
            return False
        src = idx.get(fname.lower())
        if not src:
            return False
        a = d.get("2d_annotation", {})
        x, y   = float(a.get("x", 0)), float(a.get("y", 0))
        bw, bh = float(a.get("width", 0)), float(a.get("height", 0))
        if bw <= 0 or bh <= 0:
            return False
        cx, cy, nw, nh = (x+bw/2)/iw, (y+bh/2)/ih, bw/iw, bh/ih
        if not all(0 < v <= 1 for v in [cx, cy, nw, nh]):
            return False
        stem    = Path(fname).stem
        dst_img = Path(out_img) / f"{stem}.jpg"
        if dst_img.exists():
            return False   # 중복 스킵
        img = cv2.imread(src)
        if img is None:
            return False
        cv2.imwrite(str(dst_img),
                    cv2.resize(img, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA),
                    [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        (Path(out_lbl) / f"{stem}.txt").write_text(
            f"{cls2id[fc]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n", encoding="utf-8")
        return True
    except Exception:
        return False

def _process_split(label_dir, idx, cls2id, out_base, split):
    jsons = list(Path(label_dir).rglob("*.json"))
    print(f"\n[{split}] {len(jsons):,}개 변환 중...")
    out_img = Path(out_base) / split / "images"
    out_lbl = Path(out_base) / split / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    ok = skip = 0
    for i, jf in enumerate(jsons):
        if i % 500 == 0:
            print(f"  {i/len(jsons)*100:.1f}% ({i:,}/{len(jsons):,}) ok:{ok:,}", end="\r")
        if _convert_one(str(jf), idx, cls2id, str(out_img), str(out_lbl)):
            ok += 1
        else:
            skip += 1
    print(f"\n  [{split}] 완료 — 성공:{ok:,} / 실패:{skip:,}")
    return ok

# ── 실행 ──────────────────────────────────────────────────────────────────────
print("=" * 50)
print(f"전처리 시작: {TS_ZIP_NAME}")
print("=" * 50)

print("\n[1] 클래스 수집")
all_classes, cls2id = _collect_classes(DATASET_DIR)

print("\n[2] 이미지 인덱스")
idx = _build_index(TS_EXTRACT_LOCAL)

# TL 라벨 경로 자동 탐지
tl_candidates = [
    Path(DATASET_DIR) / "Training" / "02.라벨링데이터" / "TL",
    Path(DATASET_DIR) / "TL",
    Path(DATASET_DIR),
]
tl_dir = next(
    (str(p) for p in tl_candidates if p.exists() and list(p.rglob("*.json"))),
    DATASET_DIR
)
print(f"\n[3] train 변환  (라벨: {tl_dir})")
train_n = _process_split(tl_dir, idx, cls2id, FINETUNE_DIR, "train")

vl_candidates = [
    Path(DATASET_DIR) / "Validation" / "02.라벨링데이터" / "VL",
    Path(DATASET_DIR) / "VL",
]
vl_dir = next((str(p) for p in vl_candidates if p.exists() and list(p.rglob("*.json"))), None)
if vl_dir:
    print(f"\n[4] val 변환  (라벨: {vl_dir})")
    val_n = _process_split(vl_dir, idx, cls2id, FINETUNE_DIR, "val")
else:
    val_n = 0
    print("\n[4] val 라벨 없음 — 스킵")

# data.yaml 갱신
with open(f"{FINETUNE_DIR}/data.yaml", "w", encoding="utf-8") as _f:
    yaml.dump(
        {"train": f"{FINETUNE_DIR}/train/images",
         "val":   f"{FINETUNE_DIR}/val/images",
         "nc":    len(all_classes),
         "names": all_classes},
        _f, allow_unicode=True, default_flow_style=False
    )

print(f"\n완료 — train:{train_n:,} / val:{val_n:,} / 클래스:{len(all_classes)}종")
print("Drive finetune_dataset 누적 완료")

## Step 5 — 정리 및 현황 확인

In [ ]:
import random

VAL_RATIO = 0.2
SEED      = 42

train_img_dir = Path(FINETUNE_DIR) / "train" / "images"
train_lbl_dir = Path(FINETUNE_DIR) / "train" / "labels"
val_img_dir   = Path(FINETUNE_DIR) / "val"   / "images"
val_lbl_dir   = Path(FINETUNE_DIR) / "val"   / "labels"

train_imgs = list(train_img_dir.glob("*.jpg"))
val_imgs   = list(val_img_dir.glob("*.jpg"))

total      = len(train_imgs) + len(val_imgs)
target_val = int(total * VAL_RATIO)
need_move  = target_val - len(val_imgs)

print(f"현재 — train:{len(train_imgs):,} / val:{len(val_imgs):,} / 전체:{total:,}")
print(f"목표 val 20% → {target_val:,}장, 추가 이동 필요: {need_move:,}장")

if need_move > 0:
    random.seed(SEED)
    to_move = random.sample(train_imgs, min(need_move, len(train_imgs)))
    moved = 0
    for img_path in to_move:
        lbl_path = train_lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue
        shutil.move(str(img_path), val_img_dir / img_path.name)
        shutil.move(str(lbl_path), val_lbl_dir / lbl_path.name)
        moved += 1
    print(f"이동 완료: {moved:,}장")
else:
    print("이미 val 비율 충족 — 스킵")

tr_final = len(list(train_img_dir.glob("*.jpg")))
vl_final = len(list(val_img_dir.glob("*.jpg")))
print(f"최종 — train:{tr_final:,} / val:{vl_final:,} (비율 {vl_final/(tr_final+vl_final)*100:.1f}%)")

## Step 4.5 — train/val 80:20 분리 (누적 비율 유지)

In [ ]:
# 로컬 임시 폴더 삭제
shutil.rmtree(TS_EXTRACT_LOCAL, ignore_errors=True)
print(f"임시 폴더 삭제: {TS_EXTRACT_LOCAL}")

!df -h /content | tail -1

tr = len(list(Path(f"{FINETUNE_DIR}/train/images").rglob("*.jpg"))) if Path(f"{FINETUNE_DIR}/train/images").exists() else 0
vl = len(list(Path(f"{FINETUNE_DIR}/val/images").rglob("*.jpg")))   if Path(f"{FINETUNE_DIR}/val/images").exists() else 0
print(f"Drive 누적 — train:{tr:,}장 / val:{vl:,}장")
print()
print("다음 분할 처리하려면:")
print("  Step 2 셀로 돌아가서 TS_ZIP_NAME 변경 후 Step 2~5 재실행")
print()
print("모든 분할 완료 후:")
print("  phase5_finetune_colab.ipynb Step 3(data.yaml 확인) -> Step 4(학습) 실행")